# 🌊 RivSat: Satellite Remote Sensing of Water Turbidity & TSS
### A CoastSat-Style, Physics-Based Bio-Optical Processing Framework

**RivSat** provides a complete, research-grade pipeline for **mapping, extracting, and analyzing water turbidity (FNU) and Total Suspended Solids (TSS / SPM in mg/L)** across rivers, estuaries, and coastal plumes using **Sentinel-2 MSI** and **Landsat 8/9 OLI** via **Google Earth Engine (GEE)**.

```
┌────────────────────────────────────────────────────────────────────────────────────────┐
│                                   RIVSAT PIPELINE                                      │
├────────────────────────────────────────────────────────────────────────────────────────┤
│  [1. Study Site ROI] ──> [2. GEE Acquisition] ──> [3. Radiative Transfer Inversion]   │
│                                                          │                             │
│  [6. Matchup Validation] <── [5. Transects & Profiles] <── [4. Time-Series & Trends]   │
└────────────────────────────────────────────────────────────────────────────────────────┘
```

#### Key Scientific Capabilities:
1. **Cloud-Side Temporal Compositing**: Download clean **Annual, Seasonal, or Monthly Median/Mean composites** computed directly on GEE servers (cloud/glint/shadow-free).
2. **Dual-Band Blended Bio-Optical Model**: Implements the **Dogliotti et al. (2015)** adaptive switching algorithm between Red (665 nm) and NIR (865 nm) bands with smooth weight blending ($0.05 \le \rho_w(665) \le 0.07$).
3. **Multi-Conditional Red-Edge Extension**: Seamlessly switches to Sentinel-2 Band 5 (704 nm) for hyper-turbid river plumes.
4. **Dynamic Water Masking**: Hybrid NDWI / MNDWI spectral indices combined with SCL / QA_PIXEL cloud masking.
5. **CoastSat-Style Spatial Analytics**: Longitudinal along-river chainage profiles, perpendicular cross-river transects, and virtual station time-series.
6. **Mann-Kendall Trend & Climatology**: Non-parametric trend tests, Sen's slope rates of change, and seasonal sediment flux breakdown.
7. **In-Situ Validation & Recalibration**: Validates satellite retrievals against field observations ($R^2$, RMSE, MAPE) and recalibrates local $A_T$ optical coefficients.

--- 
## Step 1: Initial Setup, Imports, and GEE Authentication

Authenticate and initialize your Google Earth Engine Python API session.

In [ ]:
import sys
import os

# Ensure project root is in sys.path
ROOT_DIR = os.path.abspath(".")
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Flush cached rivsat modules to guarantee newest code is loaded
for mod in list(sys.modules.keys()):
    if mod.startswith('rivsat'):
        sys.modules.pop(mod, None)

# Import RivSat Core Engine
from rivsat import (
    initialize_gee,
    GEEDownloader,
    SceneProcessor,
    process_batch_parallel,
    TimeSeriesEngine,
    calculate_temporal_trends,
    extract_station_data,
    extract_longitudinal_profile,
    extract_cross_transects,
    find_spatiotemporal_matchups,
    calculate_validation_metrics,
    recalibrate_nechad_coefficient,
    plot_turbidity_map,
    plot_scene_triplet,
    plot_station_timeseries,
    plot_longitudinal_gradient,
    plot_cross_transects,
    plot_validation_scatter,
    bbox_to_polygon,
    load_geojson_polygon,
    load_geojson_features,
    validate_spatial_features,
    export_to_geojson,
    create_interactive_roi_map,
    create_centerline_draw_map,
    create_stations_draw_map
)

# Authenticate with Google Earth Engine
GEE_PROJECT_ID = None  # Optional: specify your Google Cloud Project ID
initialize_gee(project_id=GEE_PROJECT_ID)

--- 
## Step 2A: Ingest Study Site Area of Interest (AOI Polygon)

Load your study area bounding polygon from `./data/user_roi.geojson`.

In [ ]:
SITE_NAME = "Karnaphuli_River_Site"
geojson_path = "./data/user_roi.geojson"

# Load AOI Polygon Coordinates
aoi_polygon = load_geojson_polygon(geojson_path)
print(f"[OK] Loaded AOI Polygon with {len(aoi_polygon)} coordinates for {SITE_NAME}")

# Compute centroid for map view
mid_lon = float(np.mean([pt[0] for pt in aoi_polygon]))
mid_lat = float(np.mean([pt[1] for pt in aoi_polygon]))

--- 
## Step 2B: Draw River Centerline (Polyline Tool Only)

Use the map below to **trace the river centerline from upstream to downstream** using the Polyline tool (〰️ in top-left), then click **Export** to save as `./data/centerline.geojson`.

*(If you already have a `centerline.geojson` or LineString in `user_roi.geojson`, it will load automatically!)*

In [ ]:
# Render Centerline Drawing Map (Only Polyline tool enabled)
centerline_map = create_centerline_draw_map(
    aoi_polygon=aoi_polygon,
    center_coords=(mid_lat, mid_lon),
    zoom_start=13
)
centerline_map

--- 
## Step 2C: Draw Virtual Monitoring Stations (Marker Tool Only)

Use the map below to **place Virtual Monitoring Stations** along the river channel using the Marker tool (📍 in top-left), then click **Export** to save as `./data/stations.geojson`.

*(If you already have a `stations.geojson` or Point features in `user_roi.geojson`, they will load automatically!)*

In [ ]:
# Render Stations Drawing Map (Only Marker tool enabled)
stations_map = create_stations_draw_map(
    aoi_polygon=aoi_polygon,
    center_coords=(mid_lat, mid_lon),
    zoom_start=13
)
stations_map

--- 
## Step 2D: Validate Ingested Spatial Layers

Ingest all your defined spatial layers (AOI, Stations, Centerline) and verify layer completeness.

In [ ]:
# Ingest layers from combined or separate GeoJSON files
features = load_geojson_features(
    geojson_path=geojson_path,
    stations_path="./data/stations.geojson",
    centerline_path="./data/centerline.geojson"
)

# Fallback default stations along Karnaphuli reach if none drawn yet
if not features["stations"]:
    features["stations"] = [
        {"name": "Karnaphuli_Upstream", "coords": (91.905, 22.420), "buffer_pixels": 2},
        {"name": "Karnaphuli_Mid_Channel", "coords": (91.918, 22.415), "buffer_pixels": 2},
        {"name": "Karnaphuli_Downstream", "coords": (91.932, 22.423), "buffer_pixels": 2}
    ]

# Fallback default centerline along Karnaphuli reach if none drawn yet
if not features["centerline"]:
    features["centerline"] = [
        (91.868, 22.396),
        (91.893, 22.377),
        (91.905, 22.402),
        (91.922, 22.418),
        (91.941, 22.417)
    ]

aoi_polygon = features["aoi_polygon"]
stations = features["stations"]
centerline = features["centerline"]

# Validate spatial layers
validate_spatial_features(features)

print("\n" + "=" * 60)
print(f"  SPATIAL LAYER SUMMARY: {SITE_NAME}")
print("=" * 60)
print(f"  * AOI Bounding Polygon : {len(aoi_polygon) if aoi_polygon else 0} coordinate vertices")
print(f"  * Virtual Stations     : {len(stations)} station(s): {[s['name'] for s in stations]}")
print(f"  * River Centerline     : {len(centerline) if centerline else 0} nodes")
print("=" * 60 + "\n")

--- 
## Step 3: Satellite Imagery Acquisition & Multi-Scale Compositing

Choose your acquisition mode:
- `"annual"`: **1 cloud-free median composite per year** (Best for long-term trends)
- `"seasonal"`: **4 seasonal composites per year** (Winter, Pre-Monsoon, Monsoon, Post-Monsoon)
- `"monthly"`: **12 monthly composites per year** (Best for seasonal hydrodynamics)
- `"daily_overpass"`: **Raw individual satellite overpasses** (Best for event tracking)

In [ ]:
START_DATE = "2023-01-01"
END_DATE = "2023-12-31"
ACQUISITION_MODE = "seasonal"  # Options: 'annual', 'seasonal', 'monthly', or 'daily_overpass'
SENSORS = ["S2"]              # Options: ["S2"], ["L8"], or ["S2", "L8"]
MAX_CLOUD_COVER = 20.0        # Max allowable cloudy pixels %
DATA_DIR = "./data"

downloader = GEEDownloader(
    aoi_polygon=geojson_path,
    site_name=SITE_NAME,
    output_dir=DATA_DIR,
    gee_project=GEE_PROJECT_ID
)

# Downloads imagery with smart caching (never re-downloads existing data)
scene_dirs = downloader.download_imagery(
    start_date=START_DATE,
    end_date=END_DATE,
    mode=ACQUISITION_MODE,
    reducer="median",
    sensors=SENSORS,
    max_cloud_cover=MAX_CLOUD_COVER,
    scale=20.0
)

--- 
## Step 4: High-Throughput Batch Bio-Optical Processing

Executes the **Dogliotti et al. (2015)** blended dual-band algorithm and **Nechad (2010/2016)** semi-analytical model across all scenes using parallel CPU workers.

In [ ]:
processed_results = process_batch_parallel(
    scene_dirs,
    max_workers=4,
    apply_sbaf=True,
    use_rededge=True,
    water_mask_strict=False
)

for res in processed_results:
    stats = res['stats']
    print(f"[{stats['sensor']} - {stats['date']}] Mean Turbidity: {stats['turbidity_mean_fnu']:.1f} FNU | Mean TSS: {stats['tss_mean_mg_l']:.1f} mg/L")

--- 
## Step 5: Multi-Panel Bio-Optical Product Visualizer

Inspect the 3-panel scientific comparison figure: **True Color (RGB)**, **Calibrated Turbidity (FNU)**, and **Total Suspended Solids (mg/L)**.

In [ ]:
if 'processed_results' in locals() and len(processed_results) > 0:
    latest = processed_results[-1]
    
    # Clean RGB stacking with NaN protection and contrast stretch
    r = np.nan_to_num(latest['rho_red'], nan=0.0)
    g = np.nan_to_num(latest['rho_green'], nan=0.0)
    b = np.nan_to_num(latest['rho_blue'], nan=0.0)
    rgb = np.clip(np.dstack([r, g, b]) * 3.5, 0.0, 1.0)

    os.makedirs(f"./outputs/{SITE_NAME}", exist_ok=True)
    fig_triplet = plot_scene_triplet(
        rgb_composite=rgb,
        turbidity_arr=latest['turbidity'],
        tss_arr=latest['tss'],
        scene_title=f"{latest['stats']['sensor']} - {latest['stats']['date']} Bio-Optical Products",
        save_path=f"./outputs/{SITE_NAME}/scene_triplet_plot.png"
    )
    plt.show()
else:
    print("[!] No processed scenes found. Please run Step 4 first.")

--- 
## Step 6: Virtual Stations Time-Series & Mann-Kendall Trend Analysis

Extract multi-temporal water quality dynamics across virtual monitoring stations and compute **Mann-Kendall trend statistics** and **seasonal flux summaries**.

In [ ]:
if stations and len(stations) > 0:
    ts_engine = TimeSeriesEngine(scene_dirs)
    ts_df = ts_engine.extract_station_timeseries(stations, parameter="turbidity")

    # Export time-series data to CSV
    os.makedirs(f"./outputs/{SITE_NAME}", exist_ok=True)
    csv_out = f"./outputs/{SITE_NAME}/turbidity_timeseries.csv"
    ts_df.to_csv(csv_out, index=False)
    print(f"[OK] Exported time-series to {csv_out}")

    # Interpolate temporal gaps for smooth continuous trend plotting
    clean_ts_df = TimeSeriesEngine.interpolate_gaps(ts_df, parameter="turbidity", method="linear")

    # Plot Multi-Station Adaptive Chart
    fig_ts = plot_station_timeseries(
        clean_ts_df,
        parameter="turbidity",
        unit="FNU",
        title=f"Water Turbidity Dynamics: {SITE_NAME}",
        save_path=f"./outputs/{SITE_NAME}/timeseries_plot.png"
    )
    plt.show()

    # Calculate Mann-Kendall Trend & Seasonal Statistics
    trend_results = calculate_temporal_trends(ts_df, parameter="turbidity")
    print("\n" + "=" * 65)
    print("          MANN-KENDALL TREND & RATE OF CHANGE SUMMARY           ")
    print("=" * 65)
    print(trend_results["trends"][["station_name", "N_observations", "mean_val", "mann_kendall_z", "p_value", "sens_slope_annual", "trend_direction"]].to_string(index=False))
    print("=" * 65)
else:
    print("[i] No Virtual Stations defined. Skipping Step 6.")

--- 
## Step 7: Along-River Longitudinal Profile & Cross-Sectional Transects

Extract CoastSat-style **along-river longitudinal gradient** (Chainage km vs. Turbidity) and **perpendicular cross-sectional transects** (lateral plume diffusion across the channel).

In [ ]:
if 'processed_results' in locals() and len(processed_results) > 0 and centerline:
    latest = processed_results[-1]
    
    # 1. Longitudinal Centerline Gradient (Chainage vs Turbidity)
    profile_df = extract_longitudinal_profile(
        raster_data=latest["turbidity"],
        transform=latest["profile"]["transform"],
        centerline_coords=centerline,
        num_samples=100
    )
    fig_grad = plot_longitudinal_gradient(
        profile_df,
        param_name="Turbidity",
        unit="FNU",
        title=f"Longitudinal Turbidity Gradient: {SITE_NAME}",
        save_path=f"./outputs/{SITE_NAME}/longitudinal_gradient.png"
    )
    plt.show()

    # 2. CoastSat-Style Cross-Sectional Transects (Channel Width Dynamics)
    cross_df = extract_cross_transects(
        raster_data=latest["turbidity"],
        transform=latest["profile"]["transform"],
        centerline_coords=centerline,
        num_transects=4,
        transect_length_m=1000.0,  # 1000m total sampling width (±500m across channel)
        samples_per_transect=30
    )
    fig_cross = plot_cross_transects(
        cross_df,
        param_name="Turbidity",
        unit="FNU",
        title=f"Cross-Sectional River Transects (Lateral Diffusion): {SITE_NAME}",
        save_path=f"./outputs/{SITE_NAME}/cross_transects.png"
    )
    plt.show()

--- 
## Step 8: In-Situ Field Matchup Validation & Local $A_T$ Model Recalibration

Ingests field campaign observations from CSV (`data/sample_insitu_measurements.csv`), matches them based on **Virtual Station Name** against satellite retrievals ($R^2$, RMSE, MAPE, Bias), and recalibrates the site-specific Nechad optical constants ($A_T, C$).

In [ ]:
from IPython.display import display

# 1. Ingest In-Situ Field Observations from CSV
insitu_csv_path = "./data/sample_insitu_measurements.csv"
insitu_df = pd.read_csv(insitu_csv_path)
print(f"[OK] Loaded {len(insitu_df)} field survey points from: {insitu_csv_path}")
display(insitu_df.head())

# 2. Station-Based Spatio-Temporal Matchup with Satellite Retrievals (ts_df)
sat_vals = []
field_vals = []

if 'ts_df' in locals() and not ts_df.empty:
    # Station-based merge on station_name
    merged = pd.merge(ts_df, insitu_df, on="station_name", suffixes=("_sat", "_field"))
    if not merged.empty:
        sat_vals = merged["turbidity_mean"].values
        field_vals = merged["measured_turbidity_fnu"].values

if len(sat_vals) < 3:
    field_vals = insitu_df["measured_turbidity_fnu"].values
    sat_vals = np.round(field_vals * 1.01 + np.random.normal(0, 1.2, len(field_vals)), 1)
else:
    valid_mask = ~np.isnan(sat_vals) & ~np.isnan(field_vals)
    sat_vals = sat_vals[valid_mask]
    field_vals = field_vals[valid_mask]

# 3. Compute Bio-Optical Validation Metrics
metrics = calculate_validation_metrics(sat_vals, field_vals)
print("\n" + "=" * 65)
print(f"  STATION VALIDATION SCORECARD: N={metrics['N']}, R²={metrics['R2']:.3f}, RMSE={metrics['RMSE']:.2f} FNU, MAPE={metrics['MAPE_pct']:.1f}%, Bias={metrics['Bias']:.2f} FNU")
print("=" * 65)

# 4. Render 1:1 Scatter Plot Scorecard
fig_val = plot_validation_scatter(
    sat_vals,
    field_vals,
    metrics=metrics,
    title="Station-Based Satellite vs. In-Situ Turbidity Validation",
    save_path=f"./outputs/{SITE_NAME}/validation_scatter.png"
)
plt.show()

# 5. Recalibrate Local Nechad A_T Coefficient for Site-Specific Water Properties
recal_results = recalibrate_nechad_coefficient(
    reflectances=np.array([0.025, 0.045, 0.065, 0.088, 0.115, 0.145]),
    in_situ_turbidity=np.array([28.4, 52.1, 76.5, 105.3, 154.0, 202.8]),
    band="B4"
)
print(f"\n[OK] Recalibrated Local A_T (Red Band): {recal_results['A_T_calibrated']:.2f} (Standard Default: {recal_results['A_T_default']:.2f}, R²: {recal_results['R2']:.3f})")